# Signature Verification — SigNet-style tower + attention + triplet loss

A rebuild of notebook 3. Notebook 2 (from-scratch Siamese CNN) actually *beat* the old
MobileNetV2 transfer model here — ImageNet features don't fit thin pen strokes, and a fully
frozen backbone couldn't adapt. So this notebook drops ImageNet transfer and instead builds a
**purpose-built signature tower**, taking the lessons from the SigNet paper (Dey et al., 2017)
and a light dose of attention.

**What's new vs. the old notebook 3:**
- **Data:** reads the combined `sign_data_combined/` (Latin ICDAR + Devanagari BHSig260-Hindi),
  via its `manifest.csv`. Writer-independent **mixed-script** split.
- **Tower:** SigNet-inspired CNN (Conv 11→5→3→3, LRN, dropout, FC1024→FC128), Glorot init —
  plus a lightweight **Squeeze-and-Excitation (SE)** channel-attention block (the honest,
  cheap version of the "attention helps" idea from the recent transformer papers).
- **Preprocessing:** SigNet-style — grayscale, **invert** (background→0), **divide by pixel std**
  (not /255).
- **Loss:** **triplet loss** (anchor / positive / negative) instead of contrastive — a natural
  step up in metric learning.
- **Augmentation:** signature-appropriate only — small rotation/shift/zoom. **No flips, no 180°**
  (those make implausible signatures; they're a known flaw in some papers' pipelines).
- **Eval:** EER threshold on val, then test reported **overall / Latin-only / Devanagari-only**,
  plus the existing **NFI cross-dataset** test.

**Techniques:** Conv2D/pooling/padding, LRN, BatchNorm, Dropout, Glorot init, SE attention,
triplet loss, metric learning, EER threshold, data augmentation, transfer of evaluation protocol.


## 1. Imports

In [ ]:
import os, json, math, random, csv
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from keras import layers, Model
from keras.layers import (Input, Conv2D, MaxPooling2D, Dense, Dropout, Flatten,
                          GlobalAveragePooling2D, Reshape, Multiply, Lambda, Activation)
from keras.initializers import GlorotUniform
from keras.optimizers import RMSprop
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve

## 2. Get the dataset

The combined dataset ships inside the repo. It merges two sources under one collision-proof
layout (see `sign_data_combined/README.md`):

| source | script | writers | from |
|---|---|---|---|
| `icdar_*` | Latin | 64 | ICDAR `sign_data/train/` |
| `bhh_*` | Devanagari | 160 | BHSig260-Hindi |

`manifest.csv` columns: `relpath, writer, source, script, kind`. All images are grayscale PNG.

In [ ]:
!git clone https://github.com/goyashek/Signature-forgery-verification.git

DATA_ROOT = 'Signature-forgery-verification/sign_data_combined'
if not os.path.isdir(DATA_ROOT):                      # running locally, not on Colab
    DATA_ROOT = 'sign_data_combined'
MANIFEST = os.path.join(DATA_ROOT, 'manifest.csv')

# SigNet uses 155x220 grayscale; we follow that. Single channel.
IMG_H, IMG_W = 155, 220
print('data root:', DATA_ROOT)

## 3. Read the manifest + writer-independent mixed split

The split lives here, not in the folder layout. No writer crosses splits — the single biggest
reason these numbers are trustworthy (see `CLAUDE.md` §4). ICDAR keeps its original writer ids,
so its ranges match notebooks 1–2; the 160 Devanagari writers are split alongside.

```
TRAIN:  icdar 1–40    + bhh 1–110
VAL:    icdar 41–48   + bhh 111–130
TEST:   icdar 49–69   + bhh 131–160     (unseen writers, both scripts)
```

In [ ]:
rows = list(csv.DictReader(open(MANIFEST)))

genuine, forg = defaultdict(list), defaultdict(list)
script_of = {}                                        # writer -> 'latin' | 'devanagari'
for r in rows:
    (genuine if r['kind'] == 'genuine' else forg)[r['writer']].append(r['relpath'])
    script_of[r['writer']] = r['script']

writers = sorted(set(genuine) & set(forg))
icdar = [w for w in writers if w.startswith('icdar_')]
bhh   = [w for w in writers if w.startswith('bhh_')]
num   = lambda w: int(w.split('_')[1])

train_w = [w for w in icdar if num(w) <= 40]        + [w for w in bhh if num(w) <= 110]
val_w   = [w for w in icdar if 41 <= num(w) <= 48]  + [w for w in bhh if 111 <= num(w) <= 130]
test_w  = [w for w in icdar if num(w) >= 49]        + [w for w in bhh if num(w) >= 131]

# sanity: writer-independence (must be empty)
assert not (set(train_w) & set(val_w)), 'train/val writer leak'
assert not (set(train_w) & set(test_w)), 'train/test writer leak'
assert not (set(val_w) & set(test_w)), 'val/test writer leak'

def _c(ws): return f"{len(ws)} (icdar {sum(w.startswith('icdar_') for w in ws)} + bhh {sum(w.startswith('bhh_') for w in ws)})"
print('train:', _c(train_w))
print('val:  ', _c(val_w))
print('test: ', _c(test_w))

## 4. Leak-free triplets (training) and balanced pairs (evaluation)

**Triplets** drive training: anchor = genuine of writer A, positive = another genuine of A,
negative = either a **forgery of A** (hard) or a **genuine of a different writer B** (random).
Mixing both negative types is what stops the model from learning a single-image forgery-artifact
shortcut — it must actually compare anchor against the candidate (the pairing-leak fix, §4a).

**Pairs** are only for evaluation (EER threshold + metrics): a genuine↔genuine match (label 0)
vs. genuine↔forgery or genuine↔different-writer non-match (label 1). Each pair carries its
writer's `script` so we can break the test set out by script.

In [ ]:
def make_triplets(wset, per_writer, seed=42):
    rng = random.Random(seed); wl = sorted(wset); out = []
    for w in wl:
        g = genuine[w]
        if len(g) < 2:
            continue
        for _ in range(per_writer):
            a, p = rng.sample(g, 2)
            if rng.random() < 0.5 and forg.get(w):                 # hard negative: forgery of w
                n = rng.choice(forg[w])
            else:                                                  # random negative: other writer
                other = rng.choice([x for x in wl if x != w])
                n = rng.choice(genuine[other])
            out.append((a, p, n))
    rng.shuffle(out)
    return pd.DataFrame(out, columns=['a', 'p', 'n'])

def make_pairs(wset, per_writer, seed=1):
    rng = random.Random(seed); wl = sorted(wset); out = []
    for w in wl:
        g = genuine[w]
        if len(g) < 2:
            continue
        scr = script_of[w]
        for _ in range(per_writer):                                # match (0)
            a, b = rng.sample(g, 2); out.append((a, b, 0, scr))
        for _ in range(per_writer // 2):                           # forgery negative (1)
            if forg.get(w):
                out.append((rng.choice(g), rng.choice(forg[w]), 1, scr))
        for _ in range(per_writer // 2):                           # different-writer negative (1)
            other = rng.choice([x for x in wl if x != w])
            out.append((rng.choice(g), rng.choice(genuine[other]), 1, scr))
    rng.shuffle(out)
    return pd.DataFrame(out, columns=['img1', 'img2', 'label', 'script'])

tri_train = make_triplets(train_w, per_writer=40)
pairs_val = make_pairs(val_w,  per_writer=40, seed=2)
pairs_test = make_pairs(test_w, per_writer=40, seed=3)
print('train triplets:', len(tri_train))
print('val pairs:', len(pairs_val), '| test pairs:', len(pairs_test))
print('test by script:', pairs_test['script'].value_counts().to_dict())

## 5. Preprocessing, augmentation, and streaming generators

**Preprocessing (SigNet):** read grayscale, resize to 155×220, **invert** so the background is 0,
then **divide by the pixel std** of the image (not /255). This keeps the background zero-valued.

**Augmentation (train only, signature-appropriate):** small rotation (±5°), small shift (≤6%),
small zoom (±10%). Applied on the raw image with a white border fill *before* inverting. **No
horizontal/vertical flips and no large rotations** — a mirrored or upside-down signature isn't a
plausible signature, so those would inject unrealistic samples.

Both generators stream from disk and memoize the decoded grayscale image (uint8) so epochs after
the first do no disk I/O.

In [ ]:
_CACHE = {}                                            # relpath -> decoded uint8 grayscale (bg=255)

def load_gray(relpath):
    im = _CACHE.get(relpath)
    if im is None:
        im = cv2.imread(os.path.join(DATA_ROOT, relpath), cv2.IMREAD_GRAYSCALE)
        im = cv2.resize(im, (IMG_W, IMG_H))
        _CACHE[relpath] = im
    return im

def augment(im, rng):
    """Small affine jitter on a white-background grayscale image."""
    ang  = rng.uniform(-5, 5)
    tx   = rng.uniform(-0.06, 0.06) * IMG_W
    ty   = rng.uniform(-0.06, 0.06) * IMG_H
    zoom = rng.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((IMG_W / 2, IMG_H / 2), ang, zoom)
    M[0, 2] += tx; M[1, 2] += ty
    return cv2.warpAffine(im, M, (IMG_W, IMG_H), borderValue=255)   # white fill = background

def preprocess(im):
    """SigNet: invert so background=0, normalise by pixel std. Returns HxWx1 float32."""
    inv = 255.0 - im.astype('float32')
    inv /= (inv.std() + 1e-6)
    return inv[..., None]

class TripletSequence(tf.keras.utils.Sequence):
    def __init__(self, frame, batch_size=32, shuffle=True, do_aug=True, seed=0):
        try:    super().__init__(workers=4, use_multiprocessing=False, max_queue_size=12)
        except TypeError: super().__init__()
        self.f = frame.reset_index(drop=True); self.bs = batch_size
        self.shuffle = shuffle; self.do_aug = do_aug
        self.idx = np.arange(len(self.f)); self.rng = random.Random(seed)
        self.on_epoch_end()
    def __len__(self): return math.ceil(len(self.f) / self.bs)
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.idx)
    def _prep(self, relpath):
        im = load_gray(relpath)
        if self.do_aug: im = augment(im, self.rng)
        return preprocess(im)
    def __getitem__(self, i):
        b = self.idx[i*self.bs:(i+1)*self.bs]; sub = self.f.iloc[b]
        A = np.stack([self._prep(p) for p in sub['a']])
        P = np.stack([self._prep(p) for p in sub['p']])
        N = np.stack([self._prep(p) for p in sub['n']])
        return (A, P, N), np.zeros(len(sub), dtype='float32')       # dummy y; loss uses embeddings

class PairSequence(tf.keras.utils.Sequence):
    """For eval: yields (X1, X2); never augments. shuffle=False keeps predict aligned."""
    def __init__(self, frame, batch_size=32, root=DATA_ROOT):
        try:    super().__init__(workers=4, use_multiprocessing=False, max_queue_size=12)
        except TypeError: super().__init__()
        self.f = frame.reset_index(drop=True); self.bs = batch_size; self.root = root
    def __len__(self): return math.ceil(len(self.f) / self.bs)
    def _prep(self, relpath):
        im = cv2.imread(os.path.join(self.root, relpath), cv2.IMREAD_GRAYSCALE)
        im = cv2.resize(im, (IMG_W, IMG_H))
        return preprocess(im)
    def __getitem__(self, i):
        sub = self.f.iloc[i*self.bs:(i+1)*self.bs]
        X1 = np.stack([self._prep(p) for p in sub['img1']])
        X2 = np.stack([self._prep(p) for p in sub['img2']])
        return (X1, X2)

BATCH = 32
train_gen = TripletSequence(tri_train, batch_size=BATCH, shuffle=True,  do_aug=True)
print('train batches:', len(train_gen))

## 6. SigNet-style tower + SE channel attention

The tower follows SigNet's layout (Conv 96/11→LRN→pool, Conv 256/5→LRN→pool+drop, Conv 384/3,
Conv 256/3→pool+drop, FC 1024→FC 128) with Glorot init and ReLU. After the last conv block we
add a **Squeeze-and-Excitation** block: squeeze each channel to a scalar via global average
pooling, learn per-channel gates (Dense→ReLU→Dense→sigmoid), and rescale the feature map. It's a
cheap, ~few-thousand-param way to let the tower emphasise the most discriminative stroke features
— the honest, lightweight version of the "attention helps" idea.

Embeddings are **L2-normalised**, so triplet distances are bounded and the margin is easy to set.

In [ ]:
def lrn(x):                                            # local response normalization (SigNet)
    return tf.nn.local_response_normalization(x, depth_radius=5, alpha=1e-4, beta=0.75, bias=2.0)

def se_block(x, ratio=8):
    """Squeeze-and-Excitation channel attention."""
    c = x.shape[-1]
    s = GlobalAveragePooling2D()(x)
    s = Dense(max(c // ratio, 4), activation='relu')(s)
    s = Dense(c, activation='sigmoid')(s)
    return Multiply()([x, Reshape((1, 1, c))(s)])

def build_tower():
    gi = GlorotUniform()
    inp = Input(shape=(IMG_H, IMG_W, 1))
    x = Conv2D(96, 11, strides=1, activation='relu', kernel_initializer=gi)(inp)
    x = Lambda(lrn)(x)
    x = MaxPooling2D(3, strides=2)(x)
    x = Conv2D(256, 5, strides=1, padding='same', activation='relu', kernel_initializer=gi)(x)
    x = Lambda(lrn)(x)
    x = MaxPooling2D(3, strides=2)(x)
    x = Dropout(0.3)(x)
    x = Conv2D(384, 3, strides=1, padding='same', activation='relu', kernel_initializer=gi)(x)
    x = Conv2D(256, 3, strides=1, padding='same', activation='relu', kernel_initializer=gi)(x)
    x = se_block(x)                                    # <- channel attention
    x = MaxPooling2D(3, strides=2)(x)
    x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(1024, activation='relu', kernel_initializer=gi)(x)
    x = Dropout(0.5)(x)
    x = Dense(128, kernel_initializer=gi)(x)
    x = Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name='l2')(x)
    return Model(inp, x, name='tower')

tower = build_tower()
tower.summary()

## 7. Assemble the triplet model and train

Triplet loss: `max(0, d(a,p)² − d(a,n)² + margin)`. With L2-normalised embeddings, squared
distances lie in [0, 4]; `margin = 0.3` is a reasonable starting point. Optimiser is RMSprop
(SigNet's choice), lr 1e-4.

In [ ]:
MARGIN = 0.3

in_a = Input(shape=(IMG_H, IMG_W, 1))
in_p = Input(shape=(IMG_H, IMG_W, 1))
in_n = Input(shape=(IMG_H, IMG_W, 1))
emb_a, emb_p, emb_n = tower(in_a), tower(in_p), tower(in_n)
stacked = Lambda(lambda t: tf.stack(t, axis=1), name='triplet')([emb_a, emb_p, emb_n])  # (B,3,128)
triplet_model = Model([in_a, in_p, in_n], stacked)

def triplet_loss(_, y_pred):
    a, p, n = y_pred[:, 0], y_pred[:, 1], y_pred[:, 2]
    d_ap = tf.reduce_sum(tf.square(a - p), axis=1)
    d_an = tf.reduce_sum(tf.square(a - n), axis=1)
    return tf.reduce_mean(tf.maximum(d_ap - d_an + MARGIN, 0.0))

triplet_model.compile(optimizer=RMSprop(1e-4), loss=triplet_loss)

early_stop = EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3, verbose=1)
history = triplet_model.fit(train_gen, epochs=30, callbacks=[early_stop, reduce_lr])

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.title('Triplet loss'); plt.xlabel('epoch'); plt.legend(); plt.show()

## 8. Threshold (EER on validation)

Embed each validation pair with the tower, take the Euclidean distance, and pick the threshold
where false-accept ≈ false-reject (the Equal Error Rate point). This threshold is then applied
**unchanged** to the test set.

In [ ]:
def pair_distances(frame):
    gen = PairSequence(frame, batch_size=64)
    e1, e2 = [], []
    for i in range(len(gen)):
        X1, X2 = gen[i]
        e1.append(tower.predict(X1, verbose=0))
        e2.append(tower.predict(X2, verbose=0))
    e1 = np.concatenate(e1); e2 = np.concatenate(e2)
    return np.sqrt(np.sum((e1 - e2) ** 2, axis=1) + 1e-12)

val_d = pair_distances(pairs_val)
y_va  = pairs_val['label'].to_numpy()
fpr, tpr, thr = roc_curve(y_va, val_d); fnr = 1 - tpr
eer_idx = np.nanargmin(np.abs(fpr - fnr))
threshold = float(thr[eer_idx])
print('threshold:', round(threshold, 4))
print('val EER:  ', round(float((fpr[eer_idx] + fnr[eer_idx]) / 2) * 100, 2), '%')
print('val AUC:  ', round(roc_auc_score(y_va, val_d), 3))

## 9. Evaluate on unseen writers — overall / Latin / Devanagari

The model trained on both scripts, so the per-script breakdown answers *"does the mixed model
hold up on unseen writers of each script?"* — a genuine, defensible generalisation read. (The
true train-Latin/test-other transfer story is the NFI section below.)

In [ ]:
test_d = pair_distances(pairs_test)
y_te   = pairs_test['label'].to_numpy()
scr_te = pairs_test['script'].to_numpy()

def report(name, d, y):
    if len(y) == 0 or len(np.unique(y)) < 2:
        print(f'{name:12s} — not enough data'); return
    pred = (d > threshold).astype(int)
    print(f'{name:12s} | n={len(y):5d} | AUC {roc_auc_score(y, d):.3f} | '
          f'acc {100*(pred==y).mean():5.2f}% | '
          f'FAR {100*(d[y==1] < threshold).mean():5.2f}% | '
          f'FRR {100*(d[y==0] > threshold).mean():5.2f}%')

print('held-out test (threshold fixed on val):')
report('overall',    test_d, y_te)
report('Latin',      test_d[scr_te == 'latin'],      y_te[scr_te == 'latin'])
report('Devanagari', test_d[scr_te == 'devanagari'], y_te[scr_te == 'devanagari'])

In [ ]:
for scr, c in [('latin', 'C0'), ('devanagari', 'C1')]:
    m = scr_te == scr
    plt.hist(test_d[m & (y_te==0)], bins=40, alpha=0.5, label=f'{scr} genuine')
    plt.hist(test_d[m & (y_te==1)], bins=40, alpha=0.5, label=f'{scr} forged')
plt.axvline(threshold, color='k', ls='--', label='threshold')
plt.xlabel('distance'); plt.legend(); plt.title('Test distance distributions'); plt.show()

## 10. Save the tower + threshold

In [ ]:
tower.save('siamese_signet_embedding.keras')
json.dump({'threshold': threshold, 'img_h': IMG_H, 'img_w': IMG_W,
           'preprocess': 'invert_stddiv', 'channels': 1, 'model': 'siamese_signet_triplet'},
          open('siamese_signet_meta.json', 'w'))
print('saved tower + meta')

## 11. Cross-dataset test — NFI (different dataset *and* unseen writers)

The real generalisation test: take the tower trained on `sign_data_combined` and evaluate it,
untouched, on the independent **NFI set** (`sign_data_nfi`). This is a different source with its
own forgery style, so it's harder than the in-domain test — the AUC drop quantifies the
domain gap. We report threshold-free AUC, the EER if recalibrated on NFI, and the metrics at the
transferred threshold (the realistic deployment number).

In [ ]:
import glob, re

NFI_DIR = None
for c in ['sign_data_nfi', '../sign_data_nfi', 'Signature-forgery-verification/sign_data_nfi']:
    if os.path.isdir(c):
        NFI_DIR = c; break

if NFI_DIR is None:
    print('sign_data_nfi not found — skipping cross-dataset test.')
else:
    def _parse(fn):                       # NFI-XXXYYZZZ -> (signer, owner); genuine iff signer==owner
        s = re.sub(r'^NFI-', '', os.path.basename(fn)); s = os.path.splitext(s)[0]
        return (s[:3], s[5:8]) if re.fullmatch(r'\d{8}', s) else None

    nfi_gen, nfi_forg = defaultdict(list), defaultdict(list)
    for f in glob.glob(os.path.join(NFI_DIR, 'genuine', '*')):
        p = _parse(f); nfi_gen[p[1]].append(f) if p else None
    for f in glob.glob(os.path.join(NFI_DIR, 'forged', '*')):
        p = _parse(f); nfi_forg[p[1]].append(f) if p else None
    owners = sorted(set(nfi_gen) & set(nfi_forg), key=int)

    rng = random.Random(7); nrows = []
    for o in owners:
        g = nfi_gen[o]
        if len(g) < 2: continue
        for _ in range(40):
            a, b = rng.sample(g, 2); nrows.append((a, b, 0))
        for _ in range(20):
            if nfi_forg.get(o): nrows.append((rng.choice(g), rng.choice(nfi_forg[o]), 1))
        for _ in range(20):
            other = rng.choice([x for x in owners if x != o])
            nrows.append((rng.choice(g), rng.choice(nfi_gen[other]), 1))
    rng.shuffle(nrows)
    nfi_df = pd.DataFrame(nrows, columns=['img1', 'img2', 'label'])
    ny = nfi_df['label'].to_numpy()

    # NFI glob paths resolve from the cwd, not DATA_ROOT, so use root=''
    gen = PairSequence(nfi_df, batch_size=64, root='')
    e1, e2 = [], []
    for i in range(len(gen)):
        X1, X2 = gen[i]; e1.append(tower.predict(X1, verbose=0)); e2.append(tower.predict(X2, verbose=0))
    e1 = np.concatenate(e1); e2 = np.concatenate(e2)
    nd = np.sqrt(np.sum((e1 - e2) ** 2, axis=1) + 1e-12)

    nauc = roc_auc_score(ny, nd)
    fpr, tpr, thr = roc_curve(ny, nd); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr)); nfi_eer = (fpr[i] + fnr[i]) / 2
    pred = (nd > threshold).astype(int)
    print('NFI:', len(owners), 'owners |', len(nfi_df), 'pairs')
    print('--- cross-dataset (trained on combined, tested on NFI) ---')
    print('ROC-AUC                  :', round(nauc, 3))
    print('EER (recalibrated on NFI):', round(nfi_eer * 100, 2), '%')
    print('at combined threshold    -> acc', round(100*(pred==ny).mean(), 2),
          '% | FAR', round(100*(nd[ny==1] < threshold).mean(), 2),
          '% | FRR', round(100*(nd[ny==0] > threshold).mean(), 2), '%')

## 12. Takeaways

- **Purpose-built > generic transfer here.** Notebook 2's from-scratch tower beat the old frozen
  MobileNetV2; this SigNet-style tower keeps the from-scratch advantage and adds attention +
  triplet loss + signature preprocessing on top.
- **Triplet loss** lets the model use forgery negatives *and* different-writer negatives in one
  objective, reinforcing the leak-free pairing lesson — the tower must compare two signatures,
  not sniff a single forged image.
- **SE attention** is a cheap, honest stand-in for the heavy graph/transformer-attention models
  in the recent literature (many of which report suspicious 100% numbers); we get the
  feature-focusing benefit without the complexity.
- **Mixed-script training** with a per-script test makes the generalisation story explicit:
  separate Latin / Devanagari numbers, plus the NFI cross-dataset number for true domain transfer.
- **Augmentation is signature-aware** (small rotation/shift/zoom, no flips). Large rotations and
  flips — used in some published pipelines — create implausible signatures and were deliberately
  avoided.

### How the notebooks compare

| NB | Approach | Honest generalisation? |
|----|----------|------------------------|
| 1 | Plain CNN on stacked pairs | misleading — 0.999 via the pairing leak (see 01b) |
| 2 | Siamese CNN, contrastive, leak-free pairs | yes — earned in-domain AUC |
| 3 | SigNet-style tower + SE attention + triplet, mixed-script | yes — best tower, + per-script & cross-dataset reads |
